In [ ]:
!pip install --upgrade spark_session_builder

## Setting up spark session 

In [3]:
# import packages
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import yaml

from pyspark.sql.functions import col,explode, when, from_json, explode_outer, lit, to_json, schema_of_json, regexp_replace, expr, concat, round
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, BooleanType

from spark_session_builder import init_spark_session


# Set dependencies
maven_packages = [
    "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.4.2",
    "org.apache.hadoop:hadoop-aws:3.3.2"
]

# Load secrets
with open('secrets.yaml', 'r') as file:
    secrets = yaml.safe_load(file)

# Maven packages and spark configurations to interact with the Data Lakehouse 
# will be provided as default when the following environment variables are set:
# - STAGE
# - REGIOM
# - COMPANY
#
# The following environment variables are also required:
# - MINIO_ACCESS_KEY
# - MINIO_SECRET_KEY
    
os.environ["STAGE"] = "dev"
os.environ["REGION"] = "nl"
os.environ["COMPANY"] = "logex"
os.environ["MINIO_ACCESS_KEY"] = secrets["MINIO_ACCESS_KEY"]
os.environ["MINIO_SECRET_KEY"] = secrets["MINIO_SECRET_KEY"]

# We only have to supply the master url, name of the application,
# and optionally, additional configuration fields, in a dictionary format.
# The default Spark config is applied automatically, and so is the pod ip address.

spark = init_spark_session(spark_master_url=os.getenv("SPARK_MASTER_URL"), app_name="fhir_poc")


INFO:root:Initializing Spark session with application name: [shreyas.patil] fhir_poc


:: loading settings :: url = jar:file:/opt/conda/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.3_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5f9a7382-d5af-4eba-bb41-c4acc32cb7d8;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.3_2.12;1.4.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.3_2.12;0.67.0 in central
:: resolution report :: resolve 811ms :: artifacts dl 20ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from centr

24/04/29 13:47:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/04/29 13:47:54 WARN SparkConf: Total executor cores: 5 is not divisible by cores per executor: 3, the left cores: 2 will not be allocated


INFO:root:Spark session successfully initialized


## Function to fully flatten any nested field

In [4]:
def _get_complex_fields(df)->dict:
    complex_fields = dict([(field.name, field.dataType)\
                           for field in df.schema.fields\
                           if type(field.dataType) == ArrayType or  type(field.dataType) == StructType])
    
    return complex_fields

def flatten(df):
    # compute Complex Fields (Lists and Structs) in Schema   
    complex_fields = _get_complex_fields(df)
   
    while len(complex_fields)!=0:
        col_name=list(complex_fields.keys())[0]

        # if StructType then convert all sub element to columns.
        # i.e. flatten structs
        if (type(complex_fields[col_name]) == StructType):
            expanded = [col(col_name+'.'+k).alias(col_name+'_'+k) for k in [ n.name for n in  complex_fields[col_name]]]
            df=df.select("*", *expanded).drop(col_name)

        # if ArrayType then add the Array Elements as Rows using the explode function
        # i.e. explode Arrays
        elif (type(complex_fields[col_name]) == ArrayType):    
            df=df.withColumn(col_name,explode_outer(col_name))

        # recompute remaining Complex Fields in Schema       
        complex_fields = _get_complex_fields(df)
    return df

In [5]:
import time
start_time = time.time()

## Read in FHIR bundle and extract Resource of Interest

In [6]:
df = spark.read.json("s3a://fhir-poc-tabular-data/dhfa_test_data.json",  multiLine=True)
df_2 = df.withColumn("resource_list", F.explode("entry")).select("resource_list")
res_filter = df_2["resource_list.resource.resourceType"] == "Observation"
res_df = df_2.filter(res_filter)

res_df = res_df \
.select(col("resource_list.fullUrl").alias("full_url"), col("resource_list.resource").alias("res"))

json_column = to_json(col("res"))
res_df = res_df.withColumn("json_column", json_column).select(col("json_column"))
res_df.show(n=1, truncate=False, vertical=True)

24/04/29 13:48:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


24/04/29 13:48:14 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 json_column | {"code":"{\"coding\":[{\"code\":\"UMCN#4645\",\"display\":\"\",\"system\":\"http://radboudumc.nl/fhir/attributekey\"}]}","context":{"reference":"Encounter/f84bd6d3-e839-4a5a-ada3-aa0ce5a3471c"},"id":"41a070be-7254-4c44-9faf-c5a6a754eb91","issued":"2021-02-

## Clean up json 

In [7]:
patterns = [
    (r'"\{', '{'),   # Replace "{  with {
    (r'\}"', '}'),   # Replace  }" with }
    (r'"\[', '['),   # Replace "[ with [
    (r'\]"', ']'),   # Replace ]" with ]
    (r'\\', ''),     # Remove backslashes
    (r'="', '='),    # Remove the " within the xml tag div's value
    (r'">', '>')     # Remove the " within the xml tag div's value
]

# Apply regexp_replace to replace the patterns in the DataFrame column
for pattern, replacement in patterns:
    res_df = res_df.withColumn("json_column", regexp_replace(col("json_column"), pattern, replacement))

## Generate reference Spark schema for resource Type from sample FHIR resource
#### Sample FHIR resource here is generated from and example generated using SIMPLIFIER profile (https://simplifier.net/palga-logex-r4/palgaobservation)
#### NOTE : Some fields might contain variable data type denoted by [x] next to field  in FHIR and its important to have all variations of the fields value to ensure there is no data loss 

In [8]:
schema = spark.read.json("s3a://fhir-poc-tabular-data/generic_observation_sample_schema.json",  multiLine=True)
schema.printSchema()

root
 |-- basedOn: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- display: string (nullable = true)
 |    |    |-- reference: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- bodySite: struct (nullable = true)
 |    |-- coding: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- code: string (nullable = true)
 |    |    |    |-- display: string (nullable = true)
 |    |    |    |-- system: string (nullable = true)
 |    |    |    |-- userSelected: string (nullable = true)
 |    |    |    |-- version: string (nullable = true)
 |    |-- text: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- coding: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- display: string (nullable = true)
 |    |  

In [9]:
schema = spark.read.json("s3a://fhir-poc-tabular-data/generic_observation_sample_schema.json",  multiLine=True).schema

  
json_schema = StructType([
    (StructField(field.name, StringType(), True) if
     type(field.dataType) == ArrayType or type(field.dataType) == StructType
     else StructField(field.name, field.dataType, True))
    for field in schema.fields
])


## Extract json and apply schema to get all topmost level fields in StringType format

In [10]:
# Apply the schema to parse the JSON data
res_df = res_df.withColumn('raw', from_json(col("json_column"), json_schema)).select(col("raw.*"))
res_df.printSchema()

root
 |-- basedOn: string (nullable = true)
 |-- bodySite: string (nullable = true)
 |-- category: string (nullable = true)
 |-- code: string (nullable = true)
 |-- component: string (nullable = true)
 |-- dataAbsentReason: string (nullable = true)
 |-- derivedFrom: string (nullable = true)
 |-- device: string (nullable = true)
 |-- effectiveDateTime: string (nullable = true)
 |-- encounter: string (nullable = true)
 |-- focus: string (nullable = true)
 |-- hasMember: string (nullable = true)
 |-- id: string (nullable = true)
 |-- implicitRules: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- issued: string (nullable = true)
 |-- language: string (nullable = true)
 |-- meta: string (nullable = true)
 |-- method: string (nullable = true)
 |-- note: string (nullable = true)
 |-- partOf: string (nullable = true)
 |-- performer: string (nullable = true)
 |-- referenceRange: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- specimen: strin

## Cast topmost fields to corresponding correct spark types based on reference schema

In [11]:
res_df_2 = res_df

for field in schema.fields:
    
    field_name = field.name
    field_type = field.dataType

    if type(field_type) in [StructType, ArrayType]:
        res_df_2 = res_df_2.withColumn(
            f'parsed_{field_name}',
            from_json(col(field_name), field_type)
        ).select('*').drop(col(field_name))
        
    else:
        res_df_2 = res_df_2.withColumn(
            f'parsed_{field_name}',
            (col(field_name))
        ).select('*').drop(col(field_name))
        

new_columns = [col(column).alias(column[7:]) if column.startswith('parsed_') else col(column) for column in res_df_2.columns]
res_df_2 = res_df_2.select(new_columns)

flattened_df = flatten(res_df_2)
flattened_df.show(n=1, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------------------------------------------------
 effectiveDateTime                              | null                                                        
 id                                             | 41a070be-7254-4c44-9faf-c5a6a754eb91                        
 implicitRules                                  | null                                                        
 issued                                         | 2021-02-01T00:00:00+01:00                                   
 language                                       | null                                                        
 resourceType                                   | Observation                                                 
 status                                         | final                                                       
 valueString                                    | null                                                        
 

## Create primary key by concatenating id attribute of FHIR resource with a increasing number

In [12]:
# Creating a primary key by adding row number to each row within each partition where each partition is created from id attribute 
# of resource
window_spec = Window.partitionBy("id").orderBy("id")

# Add a row number column within each partition
flattened_df = flattened_df.withColumn("row_num", row_number().over(window_spec))
flattened_df = flattened_df.withColumn("id_pk", concat(col("row_num").cast("string"), lit("-"), col("id").cast("string")))
flattened_df = flattened_df.drop("row_num")

print(flattened_df.count())

24/04/29 13:48:55 WARN DAGScheduler: Broadcasting large task binary with size 1257.3 KiB


918


## Persist flattened dataframe in Iceberg format tracked Nessie

In [13]:
resource = "observation"
database_name = "fhir_poc"
table_name = f"dhfa_{resource}"
spark.sql(f"CREATE DATABASE IF NOT EXISTS nessie.{database_name} LOCATION 's3a://fhir-poc-tabular-data/'")


DataFrame[]

In [14]:
# Save a Spark DataFrame in an Iceberg table and register a table for SQL operations
flattened_df.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(f"nessie.{database_name}.{table_name}")

In [15]:
end_time = time.time()
print(f"Number of Resources Processed : {flattened_df.count()}")

execution_time = end_time - start_time
print(f"Execution time of script : {execution_time/60} minutes")

24/04/29 13:49:12 WARN DAGScheduler: Broadcasting large task binary with size 1257.3 KiB


Number of Resources Processed : 918
Execution time of script : 1.2041956384976704 minutes


# Queries 

In [16]:
 # Read an Iceberg table into a Spark DataFrame
df = spark.read.table(f"nessie.{database_name}.{table_name}")
df.show(1,  truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------------------------------------------------
 effectiveDateTime                              | null                                                        
 id                                             | 0062b545-e0dc-4ad6-a59a-49bf652b3c81                        
 implicitRules                                  | null                                                        
 issued                                         | 2021-01-11T00:00:00+01:00                                   
 language                                       | null                                                        
 resourceType                                   | Observation                                                 
 status                                         | final                                                       
 valueString                                    | null                                                        
 